In [ ]:
#=================================================
# Cellule "0" Explain Notebook LLM Visual Explorer
#=================================================

Pipeline

Scenario      (Cell 3)  Load the default scenario from config.py
    ↓
Embeddings   (Cell 4)  Compute embedding vectors
    ↓
Projection   (Cell 5)  Project embeddings into a 3D PCA space
    ↓
Similarity   (Cell 6)  Compute cosine similarities
    ↓
DataFrame    (Cell 7)  Build the dataframe for visualization
    ↓
Visualization(Cell 8)  Display the interactive 3D scene

In [78]:
# ==========================================================
# Cellule 1 - Initialization / environment
# ==========================================================

# Install required libraries
!pip -q install sentence-transformers plotly scikit-learn

In [79]:
# ==========================================================
# Cellule 1 bis - Update from GitHub repository
# ==========================================================

%cd /content/LLM-Visual-Explorer

!git pull

/content/LLM-Visual-Explorer
Already up to date.


In [86]:
#=================================================
# Cellule 2 - Initialization / imports
#=================================================

# Libraries only used by Notebook
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Clone the project (only if not already present)
import os

if not os.path.exists("/content/LLM-Visual-Explorer"):
    !git clone https://github.com/TintinDeBrest/LLM-Visual-Explorer.git

# Make the package visible to Python
import sys

PROJECT_DIR = "/content/LLM-Visual-Explorer"

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)


# Reload LlmExpl (only in dev phase - delete later)
import importlib

import explorer.embeddings
import explorer.projections
import explorer.plotting
import explorer.scenarios
import explorer.dataframe
import explorer.similarities

importlib.reload(explorer.embeddings)
importlib.reload(explorer.projections)
importlib.reload(explorer.plotting)
importlib.reload(explorer.scenarios)
importlib.reload(explorer.dataframe)
importlib.reload(explorer.similarities)

# Import LlmExpl functions

from explorer.config import DEFAULT_SCENARIO
from explorer.scenarios import load_scenario
from explorer.embeddings import compute_embeddings
from explorer.projections import compute_pca
from explorer.plotting import plot_scene
from explorer.dataframe import create_dataframe
from explorer.similarities import compute_similarity
from explorer.similarities import rank_similarity_pairs

In [81]:
#=================================================
# Cellule 3 - Scenario
#=================================================

words = load_scenario(DEFAULT_SCENARIO)

In [82]:
#=================================================
# Cellule 4 - Embeddings
#=================================================

scenario = load_scenario(DEFAULT_SCENARIO)

title = scenario["titre"]
words = scenario["mots"]
categories = scenario["categories"]

embeddings = compute_embeddings(words)

Loading model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [83]:
#=================================================
# Cellule 5 - Projection
#=================================================

xyz, pca = compute_pca(embeddings)


In [84]:
#=================================================
# Cellule 6 - Dataframe
#=================================================

df = create_dataframe(
    words,
    categories,
    xyz,
)

In [87]:
#================================================
# Cellule 7 - Similarities
#================================================

similarities = compute_similarity(embeddings)

# Similarity matrix
display(
    pd.DataFrame(
        np.round(similarities, 2),
        index=words,
        columns=words
    )
)

# From more to less similar
pairs = rank_similarity_pairs(
    words,
    similarities
)

for word1, word2, score in pairs:
    print(f"{word1:10s} - {word2:10s} : {score:.2f}")

,roi,reine,homme,femme,pomme,orange
roi,1.00,0.68,0.62,0.35,0.30,0.35
reine,0.68,1.00,0.63,0.79,0.42,0.39
homme,0.62,0.63,1.00,0.70,0.41,0.37
femme,0.35,0.79,0.70,1.00,0.38,0.33
pomme,0.30,0.42,0.41,0.38,1.00,0.68
orange,0.35,0.39,0.37,0.33,0.68,1.00


reine      - femme      : 0.79
homme      - femme      : 0.70
pomme      - orange     : 0.68
roi        - reine      : 0.68
reine      - homme      : 0.63
roi        - homme      : 0.62
reine      - pomme      : 0.42
homme      - pomme      : 0.41
reine      - orange     : 0.39
femme      - pomme      : 0.38
homme      - orange     : 0.37
roi        - femme      : 0.35
roi        - orange     : 0.35
femme      - orange     : 0.33
roi        - pomme      : 0.30


In [88]:
#==============================================
# Cellule 8 - Visualisation
#================================================
fig = plot_scene(df, title)
fig.show()